In [ ]:
# Shared Data Preparation
"""
This notebook prepares a fixed, stratified 85/15 train-validation split (seed = 42) for all three models.

## Instructions

1. Run this notebook once to generate the shared datasets.
2. All generated files are saved to Google Drive under `IE4483 materials/processed_data/`.
3. Each model notebook should load these files directly.
4. Do not create another train-validation split in your model notebook.

## Output Files
- `train_split.json` — Model training
- `val_split.json` — Model evaluation
- `test_split.json` — Final predictions (no labels)

"""



# Code to Load the Data
"""
from google.colab import drive
import pandas as pd

drive.mount("/content/drive")

SHARED_DIR = "/content/drive/MyDrive/Colab Notebooks/IE4483 materials/processed_data"

train = pd.read_json(f"{SHARED_DIR}/train_split.json")
val = pd.read_json(f"{SHARED_DIR}/val_split.json")
test = pd.read_json(f"{SHARED_DIR}/test_split.json")
"""

In [ ]:
# ==================== GOOGLE DRIVE ====================
from google.colab import drive
import os

drive.mount('/content/drive')

BASE_DIR = "/content/drive/MyDrive/Colab Notebooks/IE4483 materials"
SHARED_DIR = f"{BASE_DIR}/processed_data"

os.makedirs(SHARED_DIR, exist_ok=True)

print("Google Drive mounted successfully.")

In [ ]:
# ==================== SETUP ====================
import random
import numpy as np
import pandas as pd

SEED = 42
TRAIN_RATIO = 0.85

random.seed(SEED)
np.random.seed(SEED)

print(f"Random seed: {SEED}")
print(f"Train ratio: {TRAIN_RATIO}")

In [ ]:
# ==================== LOAD DATA & SPLIT ====================
import json

train_path = f"{BASE_DIR}/train.json"
test_path = f"{BASE_DIR}/test.json"

with open(train_path, "r", encoding="utf-8") as f:
    train_data = json.load(f)

with open(test_path, "r", encoding="utf-8") as f:
    test_data = json.load(f)

df_train_all = pd.DataFrame(train_data)
df_test = pd.DataFrame(test_data)

from sklearn.model_selection import train_test_split

# Stratified 85/15 split shared across all models
df_train, df_val = train_test_split(
    df_train_all,
    train_size=TRAIN_RATIO,
    stratify=df_train_all["sentiments"],
    random_state=SEED
)

df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)

print(f"Training samples: {len(df_train)}")
print(f"Validation samples: {len(df_val)}")
print(f"Test samples: {len(df_test)}")


# ==================== CLASS DISTRIBUTION VERIFICATION ====================
distribution = pd.DataFrame({
    "Original (%)": df_train_all["sentiments"].value_counts(normalize=True) * 100,
    "Training (%)": df_train["sentiments"].value_counts(normalize=True) * 100,
    "Validation (%)": df_val["sentiments"].value_counts(normalize=True) * 100
})

distribution = distribution.rename(index={0: "Negative", 1: "Positive"})
distribution = distribution.round(2)

print("\n=== Stratified Split Verification ===")
display(distribution)

positive_gap = abs(
    distribution.loc["Positive", "Training (%)"]
    - distribution.loc["Positive", "Validation (%)"]
)

print(f"\nPositive ratio difference: {positive_gap:.2f} percentage points")

assert positive_gap < 1.0, "Class distribution difference is too large!"
print("PASS: Class distributions are consistent.")


In [ ]:
# ==================== SAVE SHARED DATA ====================

df_train.to_json(
    f"{SHARED_DIR}/train_split.json",
    orient="records",
    force_ascii=False
)

df_val.to_json(
    f"{SHARED_DIR}/val_split.json",
    orient="records",
    force_ascii=False
)

df_test.to_json(
    f"{SHARED_DIR}/test_split.json",
    orient="records",
    force_ascii=False
)

print("Shared datasets saved successfully!")
print(f"Location: {SHARED_DIR}")

for filename in [
    "train_split.json",
    "val_split.json",
    "test_split.json"
]:
    path = f"{SHARED_DIR}/{filename}"
    assert os.path.exists(path), f"Missing file: {path}"
    print(f"Verified: {filename}")